# 23 | LangChain Middleware：清理旧工具结果，别让上下文越滚越胖

Agent 调工具，通常不是调一次就结束。

它可能先查订单，再查库存，再查权限，再查日志。每次工具都会把结果塞进上下文。几轮下来，模型面前就摆着一桌旧报告。

问题是：最后真正有用的，往往只有最新那份。

这篇讲 `ContextEditingMiddleware`。它不是把历史对话总结成摘要，而是专门处理另一种膨胀：**旧工具结果太多，先清掉一部分，让模型别被过期信息带偏。**

## 一、先看一个上下文污染现场

假设我们有一个需求分析 Agent，正在分析“订单报表导出”功能。

它连续查了 4 次工具：字段、权限、异常处理、数据量。前几次结果已经过期，最后一次结果才最接近当前结论。

如果不清理，模型每次回答前都要重新看一遍旧工具结果。信息是多了，脑子也容易乱了。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ClearToolUsesEdit,
    ContextEditingMiddleware,
    wrap_model_call,
)
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

load_dotenv(dotenv_path=".env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)


def build_tool_history():
    """构造一段多次工具查询后的历史消息。"""
    messages = [HumanMessage(content="请持续分析订单报表导出需求。")]

    for i in range(1, 5):
        tool_call_id = f"query_report_{i}"

        # AIMessage 里保存这次工具调用的名称和参数。
        messages.append(
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "id": tool_call_id,
                        "name": "query_report_requirements",
                        "args": {"round": i},
                    }
                ],
            )
        )

        # ToolMessage 里保存工具返回结果。这里故意写长一点，模拟真实 API / RAG 返回内容。
        messages.append(
            ToolMessage(
                name="query_report_requirements",
                tool_call_id=tool_call_id,
                content=(
                    f"第 {i} 次查询结果："
                    + "订单报表字段、权限规则、导出格式、异常处理、数据量评估。" * 80
                ),
            )
        )

    messages.append(HumanMessage(content="请基于最新查询结果，整理当前最可靠的结论。"))
    return messages


@wrap_model_call
def show_request_messages(request, handler):
    """打印真正发给模型的消息。ContextEditingMiddleware 改的就是 request.messages。"""
    print(f"[实际发给模型的消息数量] {len(request.messages)}")
    for index, message in enumerate(request.messages, start=1):
        name = getattr(message, "name", "") or ""
        content = str(message.content).replace("\n", " ")
        print(f"{index}. {message.__class__.__name__} {name}: {content[:120]}")
    return handler(request)

先不加清理，只打印模型实际收到的消息。

In [ ]:
no_cleanup_agent = create_agent(
    model=model,
    tools=[],
    middleware=[show_request_messages],
    system_prompt="你是需求分析 Agent。请根据当前上下文整理结论。",
)

no_cleanup_result = no_cleanup_agent.invoke({"messages": build_tool_history()})

print("\n[最终回复]")
print(no_cleanup_result["messages"][-1].content)

你会看到多个 `ToolMessage` 都还在上下文里。

这就是问题：旧工具结果没有消失，模型可能同时看到第 1 次、第 2 次、第 3 次、第 4 次查询结果。它不一定知道哪个才是最新有效版本。

## 二、用 ContextEditingMiddleware 清场

现在加上 `ContextEditingMiddleware`。

这里用 `ClearToolUsesEdit`：当上下文达到一定 token 阈值后，把旧工具结果替换成占位符，只保留最近的工具结果。

In [ ]:
cleanup_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    # 达到这个 token 阈值后开始清理。示例里故意设置低一点，方便看到效果。
                    trigger=120,
                    # 最近 1 个工具结果保留原文，其余旧工具结果替换成 placeholder。
                    keep=1,
                    placeholder="[旧工具结果已清理]",
                )
            ]
        ),
        # 必须放在 ContextEditingMiddleware 后面，才能看到清理后的 request.messages。
        show_request_messages,
    ],
    system_prompt=(
        "你是需求分析 Agent。请优先参考最新工具结果。"
        "如果旧工具结果已被清理，不要尝试引用其中的细节。"
    ),
)

cleanup_result = cleanup_agent.invoke({"messages": build_tool_history()})

print("\n[最终回复]")
print(cleanup_result["messages"][-1].content)

这次你会看到旧工具结果变成：

```text
ToolMessage query_report_requirements: [旧工具结果已清理]
```

最新工具结果仍然保留原文。

这不是删除整个对话，也不是让模型失忆。它只是把旧工具输出从当前模型输入里移走，给上下文腾地方。

## 三、这些参数不是装饰品

`ClearToolUsesEdit` 里最常用的是这几个参数：

| 参数 | 怎么理解 |
|---|---|
| `trigger` | 上下文达到多少 token 后开始清理 |
| `keep` | 最近几个工具结果必须保留 |
| `clear_at_least` | 至少清出多少 token，避免只清一点点 |
| `exclude_tools` | 哪些工具结果不能清，比如最终审批结论 |
| `clear_tool_inputs` | 是否连工具调用参数也清掉 |
| `placeholder` | 清理后放什么占位文本 |

一套更像生产配置的写法可以是这样：

In [ ]:
context_editing_middleware = ContextEditingMiddleware(
    edits=[
        ClearToolUsesEdit(
            trigger=3000,
            clear_at_least=1000,
            keep=2,
            # final_policy_check 这类工具可能返回最终结论，不希望被清掉。
            exclude_tools=("final_policy_check",),
            # 如果工具参数里也有大字段或敏感字段，可以考虑改成 True。
            clear_tool_inputs=False,
            placeholder="[旧工具结果已清理]",
        )
    ],
    # approximate 更快；如果模型支持精确 token 计算，也可以改成 "model"。
    token_count_method="approximate",
)

## 四、它和总结不是一回事

`SummarizationMiddleware` 和 `ContextEditingMiddleware` 虽然都是上下文治理，但处理对象不同。

| 问题 | 更适合的处理 |
|---|---|
| 长对话太多，历史决策还要保留 | `SummarizationMiddleware` 总结旧对话 |
| 工具结果太长，旧结果已经不重要 | `ContextEditingMiddleware` 清理旧工具结果 |
| 需要跨会话长期复用 | Store / 数据库 / 向量库，不要硬塞上下文 |

总结是“把旧内容压短”；清理是“旧工具结果不再给模型看”。

这两件事可以一起用，但不要混为一谈。一个管聊天历史，一个管工具结果。

## 五、哪些结果该留，哪些该清

可以按这个规则判断：

| 内容 | 建议 |
|---|---|
| 最新 API 查询结果 | 留，当前回答通常需要它 |
| 多次重复查询的旧结果 | 清，避免模型引用过期信息 |
| RAG 检索片段 | 当前问题用完即可，不一定长期留 |
| 审批结果、最终决策 | 谨慎清，必要时用 `exclude_tools` |
| 需要跨会话复用的信息 | 写入 Store / 数据库 / 向量库 |

上下文窗口不是仓库。仓库要分类上架，上下文只放这次回答真正需要的东西。

## 六、小结

`ContextEditingMiddleware` 的价值不在于“少几条消息”，而在于减少过期工具结果对模型的干扰。

可以这样记：

- 工具结果很长：考虑清理
- 最近结果重要：用 `keep` 留住
- 有些工具不能清：用 `exclude_tools`
- 工具参数也很大：看情况打开 `clear_tool_inputs`
- 旧信息以后还要用：不要靠上下文硬撑，放进持久化记忆或检索系统

下一篇继续看自定义 Middleware：当内置策略不够用时，自己拦截模型和工具调用。

参考：

- LangChain Prebuilt Middleware：https://docs.langchain.com/oss/python/langchain/middleware/built-in
- LangChain ContextEditingMiddleware API Reference：https://reference.langchain.com/python/langchain/agents/middleware/context_editing/ContextEditingMiddleware
- LangChain ClearToolUsesEdit API Reference：https://reference.langchain.com/python/langchain/agents/middleware/context_editing/ClearToolUsesEdit